# 5.1 Bayesian Inference Foundations

Belief becomes mathematics when prior knowledge and data are multiplied, normalized, and read as a posterior distribution.

Bayesian inference begins with probability, likelihood, and loss. Probability supplies the prior distribution, likelihood measures compatibility with each hypothesis, and posterior summaries become decisions under loss.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5101)


def normalize(scores):
    values = np.asarray(scores, dtype=float)
    evidence = float(values.sum())
    if evidence <= 0.0:
        raise ValueError("evidence must be positive")
    posterior = values / evidence
    return posterior, evidence


def bayes_update(prior, likelihood):
    prior = np.asarray(prior, dtype=float)
    likelihood = np.asarray(likelihood, dtype=float)
    scores = prior * likelihood
    posterior, evidence = normalize(scores)
    return posterior, evidence, scores


def beta_bernoulli_update(alpha, beta, successes, failures):
    post_alpha = alpha + successes
    post_beta = beta + failures
    mean = post_alpha / (post_alpha + post_beta)
    variance = post_alpha * post_beta / ((post_alpha + post_beta) ** 2 * (post_alpha + post_beta + 1))
    return post_alpha, post_beta, mean, variance


def beta_binomial_evidence(alpha, beta, successes, failures):
    n = successes + failures
    log_choose = math.lgamma(n + 1) - math.lgamma(successes + 1) - math.lgamma(failures + 1)
    log_beta_after = math.lgamma(alpha + successes) + math.lgamma(beta + failures) - math.lgamma(alpha + beta + n)
    log_beta_before = math.lgamma(alpha) + math.lgamma(beta) - math.lgamma(alpha + beta)
    return math.exp(log_choose + log_beta_after - log_beta_before)


def total_variation(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return 0.5 * float(np.abs(p - q).sum())


def posterior_from_counts(prior_counts, observations):
    alpha = np.asarray(prior_counts, dtype=float)
    counts = np.asarray(observations, dtype=float)
    posterior = alpha + counts
    return posterior / posterior.sum()


## The concept, built once (D1)

Bayes' rule multiplies prior belief by likelihood and divides by evidence so the result is a probability distribution.

$$p(\theta\mid D)=\frac{p(D\mid\theta)p(\theta)}{p(D)},\quad p(D)=\int p(D\mid\theta)p(\theta)d\theta$$

For a tiny two-state test model we can verify evidence by hand before reusing the same update on larger ladders.

In [ ]:

def d1_disease_update():
    prior = np.array([0.9, 0.1])
    likelihood = np.array([0.2, 0.8])
    posterior, evidence, scores = bayes_update(prior, likelihood)
    return posterior, evidence, scores


The Beta-Bernoulli shortcut is the same posterior update in pseudo-count form. The lesson numbers are $\mathrm{Beta}(2,3)+7$ successes and $3$ failures, giving $\alpha=9$, $\beta=6$, mean $0.600$, variance $0.015$, and evidence $0.079920$.

In [ ]:

posterior, evidence, scores = d1_disease_update()
post_alpha, post_beta, mean, variance = beta_bernoulli_update(2, 3, 7, 3)
bb_evidence = beta_binomial_evidence(2, 3, 7, 3)
assert post_alpha == 9
assert post_beta == 6
assert abs(mean - 0.6) < 1e-12
assert abs(variance - 0.015) < 1e-12
assert abs(bb_evidence - 0.07992007992007992) < 1e-12
assert abs(evidence - scores.sum()) < 1e-12
assert abs(posterior.sum() - 1.0) < 1e-12
print("D1 posterior", posterior)
print("Beta posterior", post_alpha, post_beta, mean, variance, bb_evidence)


## The dataset ladder

Each rung is a synthetic distribution of rising complexity: exact two-state target, four-state table, bimodal conflict, real-style click contingency, and a sparse high-dimensional categorical posterior with strong pseudo-counts.

In [ ]:

rungs = []

prior = np.array([0.9, 0.1])
likelihood = np.array([0.2, 0.8])
truth, evidence, scores = bayes_update(prior, likelihood)
rungs.append({"name": "D1 two-state disease", "truth": truth, "estimate": truth.copy(), "scores": scores})

prior = np.array([0.35, 0.25, 0.25, 0.15])
likelihood = np.array([0.12, 0.55, 0.25, 0.08])
truth, evidence, scores = bayes_update(prior, likelihood)
estimate = posterior_from_counts(prior * 40.0, np.array([2.0, 12.0, 4.0, 1.0]))
rungs.append({"name": "D2 four hypotheses", "truth": truth, "estimate": estimate, "scores": scores})

prior = np.array([0.44, 0.06, 0.06, 0.44])
likelihood = np.array([0.15, 0.45, 0.45, 0.15])
truth, evidence, scores = bayes_update(prior, likelihood)
estimate = normalize(scores + rng.normal(0.0, 0.002, size=scores.shape))[0]
rungs.append({"name": "D3 bimodal conflict", "truth": truth, "estimate": estimate, "scores": scores})

clicks = np.array([120, 90, 60, 30])
conversions = np.array([12, 6, 9, 2])
prior_counts = np.array([3.0, 3.0, 3.0, 3.0])
truth = posterior_from_counts(prior_counts, conversions)
estimate = posterior_from_counts(prior_counts, conversions * clicks / clicks.mean())
rungs.append({"name": "D4 click contingency", "truth": truth, "estimate": estimate, "scores": prior_counts + conversions})

states = np.arange(32)
prior_counts = 1.0 + 9.0 * np.exp(-states / 6.0)
observations = np.zeros(32)
observations[[3, 4, 18, 19, 20, 27]] = [1, 3, 8, 12, 7, 2]
truth = posterior_from_counts(prior_counts, observations)
estimate = posterior_from_counts(prior_counts, observations + rng.poisson(1.0, size=32))
rungs.append({"name": "D5 sparse 32-state", "truth": truth, "estimate": estimate, "scores": prior_counts + observations})

for rung in rungs:
    print(rung["name"], "states", rung["truth"].shape[0], "sample", np.round(rung["truth"][:5], 3))


In [ ]:

rows = []
for rung in rungs:
    metric = total_variation(rung["estimate"], rung["truth"])
    rows.append(metric)
    print(f"{rung['name']}: TV={metric:.6f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, rung in enumerate(rungs):
    x = np.arange(rung["truth"].shape[0])
    axes[0, i].bar(x - 0.2, rung["truth"], width=0.4, label="true")
    axes[0, i].bar(x + 0.2, rung["estimate"], width=0.4, label="estimated")
    axes[0, i].set_title(rung["name"])
    axes[0, i].set_ylim(0, max(rung["truth"].max(), rung["estimate"].max()) * 1.2)
    axes[1, i].axis("off")
axes[0, 0].legend()
axes[1, 2].plot(range(1, 6), rows, marker="o")
axes[1, 2].set_xticks(range(1, 6))
axes[1, 2].set_xlabel("rung")
axes[1, 2].set_ylabel("total variation")
axes[1, 2].set_title("TV error vs complexity")
fig.tight_layout()


## Pitfall on D5: forgetting the evidence

The product of prior pseudo-counts and data likelihood is only an unnormalized score. On D5 the raw scores do not sum to $1$; dividing by evidence fixes the posterior.

In [ ]:

d5 = rungs[-1]
wrong = d5["scores"]
wrong_sum = wrong.sum()
fixed = wrong / wrong_sum
print("wrong score sum", wrong_sum)
print("fixed probability sum", fixed.sum())
print("TV after fix", total_variation(fixed, d5["truth"]))
assert wrong_sum != 1.0
assert abs(fixed.sum() - 1.0) < 1e-12


## Evaluate it + Practice

- Metric: total-variation distance to the exact posterior on every rung.
- No-skill baseline: use the prior, unary factors, or a single local conditional without global inference.
- Cheap sanity check: posterior probabilities sum to one and reduce to normalized likelihood when the prior is uniform.
- Ablation: replace the posterior by the prior-only baseline and watch TV increase when data are informative.
- Failure signals: negative evidence, non-normalized posterior, or a strong prior silently dominating small data.

Practice prompts:
1. Change the D2 likelihood table so a different hypothesis wins.

2. Make D5 less sparse and compare TV error.

3. Compute posterior entropy for each rung.